# 🔬 ISOM 260: Open the Hood of Your Own AI

**Session 2 — How AI Actually Works** | Suffolk University | Prof. Hasan Arslan

---

Last week you **trained** a language model. It learned to invent names, and you watched its "surprise" (the loss) fall from 3.5 to 1.75.

But here's the question that should be bugging you: ***what actually changed inside?***

Today we answer it — on **your** model. We're going to open its skull and look at:

1. 🧮 **The whole brain is just numbers** — all 104,475 of them, and where they live
2. 🗺️ **The map of meaning** — your model invented its own "map of the alphabet," and it discovered something nobody told it
3. 🔮 **The crystal ball** — watch the exact probabilities behind every letter it picks
4. 👀 **Attention** — literally see where the model looks when it reads

Same rules as last week: **Shift + Enter**, top to bottom. `File → Save a copy in Drive` first.


In [ ]:
# ── Rebuild and retrain YOUR model (same code, same seed, ~2 min) ──
# This is the exact model from Session 1. Watching the loss fall should
# feel different now — today you'll see WHAT is falling into place.
import urllib.request, random
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(260); random.seed(260)

url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
names = urllib.request.urlopen(url).read().decode().splitlines()
text = "\n".join(names)
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])

block_size, n_embd, n_head, n_layer = 16, 64, 4, 2
device = "cuda" if torch.cuda.is_available() else "cpu"

def get_batch(batch_size=64):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = nn.MultiheadAttention(n_embd, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(),
                                 nn.Linear(4*n_embd, n_embd))
        self.last_attn = None   # NEW this week: we save the attention map
    def forward(self, x):
        T = x.shape[1]
        mask = torch.triu(torch.ones(T, T, device=x.device), 1).bool()
        h = self.ln1(x)
        out, weights = self.attn(h, h, h, attn_mask=mask, need_weights=True)
        self.last_attn = weights.detach()          # <- the "where am I looking" map
        x = x + out
        return x + self.mlp(self.ln2(x))

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(len(chars), n_embd)
        self.pos = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.head = nn.Linear(n_embd, len(chars))
    def forward(self, idx):
        T = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        return self.head(self.blocks(x))

model = TinyGPT().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
model.train()
for step in range(2501):
    x, y = get_batch()
    loss = F.cross_entropy(model(x).view(-1, len(chars)), y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        print(f"step {step:5d} | loss {loss.item():.3f}")
model.eval()
print("\n🧠 Your model is back. Time to open it up.")

## 🧮 Part 1 — The whole brain is just numbers

When people say a model "knows" something, where does that knowledge *live*?

Answer: in the **parameters** — the numbers that training nudged, step by step, 2,500 times. Let's count every single one and see where they sit.


In [ ]:
# ── A complete census of your model's brain ───────────────────────
total = 0
print(f"{'COMPONENT':<42}{'PARAMETERS':>12}")
print("─" * 54)
for name, p in model.named_parameters():
    label = (name
        .replace("tok.weight", "letter meanings (embeddings)")
        .replace("pos.weight", "position sense"))
    print(f"{label:<42}{p.numel():>12,}")
    total += p.numel()
print("─" * 54)
print(f"{'YOUR MODEL, ENTIRE BRAIN':<42}{total:>12,}")
print(f"\nGPT-4-class models: ~1,800,000,000,000. Same table, more rows, bigger numbers.")
print("Nothing else is in there. No rules. No dictionary. Just these numbers.")

**Sit with that for a second.** Everything your model "knows" about names — that q wants u, that names end in vowels, that 'Sufina' sounds plausible — is stored in that spreadsheet of numbers. Knowledge, as geometry.

## 🗺️ Part 2 — The map of meaning

Look at the first row of the census: `letter meanings (embeddings)`. Your model represents each of the 27 characters as a list of **64 numbers** — a point in 64-dimensional space.

Here's the magic: training moved those points around. Letters the model treats similarly drifted **close together**. Letters that behave differently drifted apart.

We can't see 64 dimensions, so we'll squash the map down to 2 (a technique called PCA) and just... look at it. **Prediction before you run this:** which letters do you think ended up neighbors?


In [ ]:
# ── Your model's map of the alphabet ──────────────────────────────
import matplotlib.pyplot as plt

emb = model.tok.weight.detach().cpu()                 # 27 letters × 64 dims
emb_centered = emb - emb.mean(dim=0)
U, S, V = torch.pca_lowrank(emb_centered, q=2)        # squash 64-D → 2-D
xy = (emb_centered @ V[:, :2]).tolist()

VOWELS = set("aeiou")
fig, ax = plt.subplots(figsize=(9, 7))
for i, ch in enumerate(chars):
    label = "·" if ch == "\n" else ch
    is_vowel = ch in VOWELS
    color = "#d62728" if is_vowel else "#1f77b4"
    ax.scatter(xy[i][0], xy[i][1], s=700, alpha=.15, color=color)
    ax.text(xy[i][0], xy[i][1], label, ha="center", va="center",
            fontsize=16, fontweight="bold", color=color)
ax.set_title("Your model's map of the alphabet\n(red = vowels — nobody told it that)", fontsize=13)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

print("The model was NEVER told what a vowel is.")
print("It grouped them because they BEHAVE alike in names — same neighborhoods, same jobs.")

### 🤯 What you're looking at

Look at **a, e, o, u** — four of the five vowels found each other. **Nobody programmed that.** The model discovered they behave alike purely from data: vowels fill the same slots, between the same kinds of letters. (Notice the common "glue" consonants like l, n, t hanging near them — they keep vowel company in real names.)

And **'i'? It wandered off on its own.** Quick discussion with your neighbor: why might 'i' behave differently from other vowels in *names* specifically? (Hint: think about where 'i' shows up — *-ia*, *-in*, *Br-i-an*.) Your model's map isn't perfect — it learned from 32,033 names, not a language. **Imperfect maps from limited data** is also exactly what's wrong with a chatbot that's only seen part of the world. More data → better maps → that's the scale story.

This is the single most important idea in modern AI: **meaning is location in space.** GPT-6 does the exact same thing, except its map has thousands of dimensions and covers words, sentences, concepts, code, and images. "King − man + woman ≈ queen" lives on that map. So does everything ChatGPT "knows."

Let's interrogate the map directly — ask your model who its neighbors are:


In [ ]:
# ── "Who do you think is similar?" — ask the model directly ───────
def neighbors(letter, k=4):
    v = emb[stoi[letter]]
    sims = F.cosine_similarity(v.unsqueeze(0), emb)
    top = sims.argsort(descending=True)[1:k+1]        # skip itself
    return [(itos[i.item()], round(sims[i].item(), 2)) for i in top]

for ch in ["a", "e", "q", "n", "z"]:
    print(f"'{ch}' is most similar to: {neighbors(ch)}")

# 🎮 YOUR TURN: change the letter below and re-run
print("\nyour pick →", "'m':", neighbors("m"))

## 🔮 Part 3 — The crystal ball

Last week we said the model "predicts the next letter." Today, let's watch it do that — with the actual probabilities on screen.

This is the exact machinery behind ChatGPT showing you a word at a time. Every word you've ever seen an AI write was sampled from a bar chart like this one.


In [ ]:
# ── The exact probabilities behind the magic trick ────────────────
def crystal_ball(prefix, topk=8):
    ctx = torch.tensor([[stoi["\n"]] + [stoi[c] for c in prefix]], device=device)
    with torch.no_grad():
        logits = model(ctx[:, -block_size:])[0, -1]
    probs = F.softmax(logits, dim=-1)
    top = probs.argsort(descending=True)[:topk]
    labels = ["end" if itos[i.item()] == "\n" else itos[i.item()] for i in top]
    values = [probs[i].item() for i in top]
    fig, ax = plt.subplots(figsize=(7, 3.2))
    bars = ax.bar(labels, values, color="#1f77b4")
    bars[0].set_color("#d62728")
    ax.set_title(f'After "{prefix}" the model expects...', fontsize=12)
    ax.set_ylabel("probability")
    plt.tight_layout(); plt.show()

crystal_ball("q")        # you know what's coming
crystal_ball("mar")      # a fork in the road: several good options
crystal_ball("jasmin")   # nearly certain how this ends

# 🎮 YOUR TURN: what prefix creates the most SUSPENSE (flattest bars)?
crystal_ball("s")

**Now temperature makes total sense:** low temperature = "always take the tallest bar" (safe, boring). High temperature = "flatten the bars and gamble" (creative, chaotic). You're not changing what the model knows — you're changing how it *bets*.

## 👀 Part 4 — Attention: watch it read

Last week's counting model had **one letter of memory**. The transformer's fix is **attention**: at every position, the model looks back at *all* previous letters and decides — with learned weights — which ones matter right now.

Your model saves those "where did I look?" maps. Let's render one.


In [ ]:
# ── The attention map: where your model looks as it reads ─────────
def show_attention(word):
    ctx = torch.tensor([[stoi["\n"]] + [stoi[c] for c in word]], device=device)
    with torch.no_grad():
        model(ctx)
    att = model.blocks[1].last_attn[0].cpu().tolist()   # layer 2's map
    labels = ["start"] + list(word)
    n = len(labels)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.imshow([row[:n] for row in att[:n]], cmap="Blues")
    ax.set_xticks(range(n)); ax.set_xticklabels(labels)
    ax.set_yticks(range(n)); ax.set_yticklabels(labels)
    ax.set_xlabel("...it looks back at this letter"); ax.set_ylabel("While reading this letter...")
    ax.set_title(f'Attention while reading "{word}"', fontsize=12)
    plt.tight_layout(); plt.show()

show_attention("sophia")

# 🎮 YOUR TURN: try your own name (lowercase, letters only)
show_attention("hasan")

**How to read it:** each row is the model processing one letter; bright cells show which earlier letters it consulted. Notice it's *not* uniform — the model learned that some history matters more than the rest. That selective look-back is the entire superpower of the transformer, and it's why "Attention Is All You Need" is the most influential AI paper of the century.

Scale this up ~10-million-fold and the same mechanism lets GPT-6 connect a pronoun on page 40 to a name on page 3.

---

## 🎓 What you now know (that most people don't)

| The magic trick | What's actually happening |
|---|---|
| "The AI knows things" | Knowledge is ~100K (or 1.8T) trained numbers — geometry, not facts |
| "It understands letters/words" | Similar things sit close together on a learned map |
| "It writes" | It samples from a probability bar chart, one token at a time |
| "It reads context" | Attention weights decide which history matters |
| "It learned" | Loss fell = numbers nudged downhill, 2,500 small steps |

### ✍️ Homework #2 — due Monday Sep 21, 12:30 PM
Post **one paragraph** (3–5 sentences) in the Canvas **Session 2** thread answering:

> *Pick the visualization that surprised you most (map / crystal ball / attention). What did it change about how you'd explain AI to a friend — or how much you'd trust an AI at work?*

**Monday:** we scale this up — how these exact mechanisms become ChatGPT (instruction tuning, RLHF), why bigger keeps getting better, and how to read AI benchmark claims without getting fooled.

*Last week you trained it. Today you understood it. Most AI users never do either.* 🔬
